In [ ]:
import altair as alt
import gcsfs
import pandas as pd

import _operator_report_utils as utils
from update_vars import (
    DIGEST_DICT, PROCESSED_GCS, 
    abbrev_month, readable_dict, analysis_month
)

alt.data_transformers.enable("vegafusion")

In [ ]:
analysis_name = "Alameda-Contra Costa Transit District"

schedule_rt_route_direction_summary_url = f"{PROCESSED_GCS}{DIGEST_DICT.schedule_rt_route_direction}_{abbrev_month}.parquet"

df = pd.read_parquet(
    schedule_rt_route_direction_summary_url,
    filesystem = gcsfs.GCSFileSystem(),
    filters=[[("Analysis Name", "==", analysis_name)]]
).reset_index(drop=True)

In [ ]:
# was in create_route_dropdown
routes_list = df.Route.unique().tolist()

route_dropdown = alt.binding_select(
    options=routes_list,
    name="Routes: ",
)

# Column that controls the bar charts
xcol_param = alt.selection_point(
    fields=["Route"], value=routes_list[0], bind=route_dropdown
)

In [ ]:
# Set drop down menu to be on the upper right for the charts
from IPython.display import HTML
display(
    HTML(
        """
<style>
form.vega-bindings {
  position: absolute;
  right: 0px;
  top: 0px;
}
</style>
"""
    )
)

In [ ]:
day_type_order = ["Weekday", "Saturday", "Sunday"]
WIDTH = 200 * 2
HEIGHT = 250

chart_scheduled_minutes = (
    alt.Chart(df)
    .mark_line()
    .encode(
        x="Date",
        y="Average Scheduled Minutes",
        color=alt.Color("Day Type:N", scale=alt.Scale(domain=day_type_order)),
        column="Direction:O",
        tooltip = ["Date", "Average Scheduled Minutes", "Route", "Direction", "Day Type"]
    ).transform_filter(xcol_param).properties(width=WIDTH, height=HEIGHT)
)

In [ ]:
chart_frequency = (
    alt.Chart(df)
    .mark_line()
    .encode(
        x="Date",
        y="Headway All Day",
        color=alt.Color("Day Type:N", scale=alt.Scale(domain=day_type_order)),
        column="Direction:O"
    ).transform_filter(xcol_param).properties(width=WIDTH, height=HEIGHT)
)

In [ ]:
chart_peak = (
    alt.Chart(df)
    .mark_line()
    .encode(
        x="Date",
        y="Headway Peak",
        color=alt.Color("Day Type:N", scale=alt.Scale(domain=day_type_order)),
        column="Direction:O"
    ).transform_filter(xcol_param).properties(width=WIDTH, height=HEIGHT)
)

In [ ]:
combined_chart = alt.vconcat(chart_scheduled_minutes, chart_frequency, chart_peak).add_params(xcol_param)
combined_chart

What if chart went with 3 day_types, then put all the metrics onto 1 chart that are related to headway, frequency.
* average scheduled minutes and headway all day are basically just 2 sides of the same coin
* headway peak and offpeak get averaged into headway all day
* see if tooltips can display
* might have to change df to long

In [ ]:
chart_scheduled_minutes2 = (
    alt.Chart(df)
    .mark_line()
    .encode(
        x="Date",
        y="Average Scheduled Minutes",
        row=alt.Row("Day Type:N", sort=day_type_order),
        column="Direction:O",
        tooltip = ["Date", "Average Scheduled Minutes", "Route", "Direction", "Day Type"]
    ).transform_filter(xcol_param).properties(width=WIDTH, height=HEIGHT)
)

#chart_scheduled_minutes2.add_params(xcol_param)

In [ ]:
chart_frequency2 = (
    alt.Chart(df)
    .mark_line()
    .encode(
        x="Date",
        y="Headway All Day",
        row=alt.Row("Day Type:N", sort=day_type_order),
        column="Direction:O"
    ).transform_filter(xcol_param).properties(width=WIDTH, height=HEIGHT)
)

In [ ]:
df.dtypes

In [ ]:
df_long = df.melt(
    id_vars = ["Date", "Analysis Name", "Day Type", "Route", "Direction"], 
    value_vars = ["Daily Trips All Day", "Frequency All Day", 
                  "Average Scheduled Minutes",
                  "Headway All Day", "Headway Peak", "Headway Offpeak"]
)

df_long["Date"] = pd.to_datetime(df_long.Date)

In [ ]:
df_long.head(2)

In [ ]:
sorted(df_long.Date.unique())

In [ ]:
legend_selection = alt.selection_point(fields = ["variable"], bind = "legend")

chart = (
    alt.Chart(df_long)
    .mark_line(point=alt.OverlayMarkDef(filled=False, fill="white"))
    .encode(
        x=alt.X("yearmonth(Date)", axis=alt.Axis(labelAngle=-45, format="%b %Y")), #tickExtra=True? how to add more space at beginning of x-axis
        y=alt.Y("value", title = ""),
        color=alt.Color("variable:N", title = "metric"), 
        row=alt.Row("Day Type:N", sort=day_type_order),
        column="Direction:O",
        tooltip = ["Date", "variable", "value", "Route", "Direction", "Day Type"],
        opacity = alt.when(legend_selection).then(alt.value(1)).otherwise(alt.value(0.2))
    ).transform_filter(xcol_param, legend_selection).properties(width=400, height=250)
    .interactive()
) 
# if use transform_filter(xcol_param, legend_selection), 
# then clicking on legend will make other lines disappear, rather than dim
# if remove it, then other lines will be dim, but the y-axis will remain shared
#  resolve_scale will not change that...resolve_scale addresses the weekday compared to sat chart

chart.add_params(legend_selection, xcol_param).resolve_scale(x="shared", y="independent").properties(
    title = "GTFS Schedule Metrics by Route"
)